In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shishiradhikari11/casmi-rdkit2025-cp313/LICENSE-NOTICE.txt
/kaggle/input/datasets/shishiradhikari11/casmi-rdkit2025-cp313/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl
/kaggle/input/datasets/aidensong123/casmi26-coconut-202609/COCONUT_ATTRIBUTION.md
/kaggle/input/datasets/aidensong123/casmi26-coconut-202609/coconut_structures.parquet
/kaggle/input/datasets/salutsirius/enveda-2m4-dataset-from-notebook02a72b3003/checkpoint2m4.pt
/kaggle/input/datasets/salutsirius/enveda-2m4-dataset-from-notebook02a72b3003/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


In [2]:
# monoisotopic masses
MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196,
            P=30.97376163, S=31.97207100, F=18.99840322, Cl=34.96885268,
            Br=78.9183371, I=126.904473, Na=22.9897692809, K=38.96370668,
            Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990
PROTON = MASS['H'] - E
H2O = 2 * MASS['H'] + MASS['O']
NH4 = MASS['N'] + 4 * MASS['H']
FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']
ACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']

ADDUCTS = {
    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E),
    "[M+Na]+": (1, 1, MASS['Na'] - E), "[M+K]+": (1, 1, MASS['K'] - E),
    "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),
    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),
    "[M+CH3OH+H]+": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),
    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),
    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O),
    "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),
    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON),
    "[M+Cl]-": (1, 1, MASS['Cl'] + E), "[M]-": (1, 1, E),
    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS['Na'] - 2 * PROTON),
    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS['Na'] - E),
    "[2M+NH4]+": (2, 1, NH4 - E), "[2M+K]+": (2, 1, MASS['K'] - E),
    "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),
    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS['Na'] - 2 * PROTON),
    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),
}

def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan)
    ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any():
            out[m] = (mz[m] * z - d) / n
    return out

In [3]:
import os, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from pathlib import Path

# !pip install rdkit
!pip install --no-index /kaggle/input/datasets/shishiradhikari11/casmi-rdkit2025-cp313/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl
# !pip install faiss-cpu
# !pip install --no-index /kaggle/input/datasets/salutsirius/faiss-cpu-notebook9e1116636c/faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import rdFingerprintGenerator
import rdkit.Chem as Chem

Processing /kaggle/input/datasets/shishiradhikari11/casmi-rdkit2025-cp313/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl


In [4]:
class SpectrumTokenizer:
    def __init__(self, mz_min=0.0, mz_max=1800.0, mz_bin_width=0.02, max_peaks=256):
        self.mz_min = mz_min
        self.mz_max = mz_max
        self.mz_bin_width = mz_bin_width
        self.max_peaks = max_peaks
        self.mz_bins = int((mz_max - mz_min) / mz_bin_width) + 1

    def __call__(self, mzs, intensities):
        mzs = np.asarray(mzs)
        intensities = np.asarray(intensities)

        # 1. cap to top-N by intensity
        if len(mzs) > self.max_peaks:
            top = np.argsort(intensities)[-self.max_peaks:]
            mzs = mzs[top]
            intensities = intensities[top]

        # 2. sort kept peaks by m/z
        order = np.argsort(mzs)
        mzs = mzs[order]
        intensities = intensities[order]

        # 3. bin m/z with clipping
        mz_idx = np.clip(
            ((mzs - self.mz_min) / self.mz_bin_width).astype(np.int64),
            0, self.mz_bins - 1
        )

        return {
            'mz_idx': torch.tensor(mz_idx, dtype=torch.long),
            'intensity': torch.tensor(intensities, dtype=torch.float32),
            'length': len(mz_idx),
        }

In [5]:
class PeakEmbedding(nn.Module):
    def __init__(self, d_model, mz_bins):
        super().__init__()
        self.mz_emb = nn.Embedding(mz_bins, d_model)
        self.intensity_proj = nn.Linear(1, d_model)

    def forward(self, mz_idx, intensity):
        # mz_idx: [B, L], intensity: [B, L]
        mz_vec = self.mz_emb(mz_idx)                       # [B, L, d]
        # intensity.unsqueeze(-1) - [B, L] -> [B, L, 1]
        # (output = input @ weight.T + bias); [B, L, d] = [B, L, 1] @ [1, d]
        int_vec = self.intensity_proj(intensity.unsqueeze(-1))  # [B, L, d] 
        
        return mz_vec + int_vec

In [6]:
class SpectrumCollator:
    def __init__(self, max_peaks=256, pad_mz_idx=0, pad_intensity=0.0):
        self.max_peaks = max_peaks
        self.pad_mz_idx = pad_mz_idx
        self.pad_intensity = pad_intensity

    def __call__(self, batch):
        # batch: list of dicts from SpectrumTokenizer, each with
        #   'mz_idx': LongTensor [len_i]
        #   'intensity': FloatTensor [len_i]
        #   plus label fields: 'smiles', 'inchikey14', etc.

        B = len(batch)
        L = self.max_peaks

        mz_idx = torch.full((B, L), self.pad_mz_idx, dtype=torch.long)
        intensity = torch.full((B, L), self.pad_intensity, dtype=torch.float32)
        mask = torch.zeros((B, L), dtype=torch.bool)

        for i, sample in enumerate(batch):
            n = min(len(sample['mz_idx']), L)
            mz_idx[i, :n] = sample['mz_idx'][:n]
            intensity[i, :n] = sample['intensity'][:n]
            mask[i, :n] = True

        out = {
            'mz_idx': mz_idx,
            'intensity': intensity,
            'mask': mask,
        }

        # pass through label fields unchanged (as lists)
        for key in batch[0]:
            if key not in ('mz_idx', 'intensity'):
                out[key] = [sample[key] for sample in batch]

        return out

In [7]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, causal=False) -> None:
    super().__init__()

    self.d_k = d_k
    self.n_heads = n_heads

    self.key = nn.Linear(d_model, d_k * n_heads)
    self.query = nn.Linear(d_model, d_k * n_heads)
    self.value = nn.Linear(d_model, d_k * n_heads)

    self.fc = nn.Linear(d_k * n_heads, d_model)

    self.causal = causal
    if causal:
      cm = torch.tril(torch.ones(max_len, max_len))
      self.register_buffer(
        'causal_mask',
        cm.view(1, 1, max_len, max_len)
    )


  def forward(self, q, k, v, pad_mask=None):
    q = self.query(q)   # (N, T, d_model) -> (N, T, h*d_k)
    k = self.key(k)   # (N, T, d_model) -> (N, T, h*d_k)
    v = self.value(v)   # (N, T, d_model) -> (N, T, h*d_k)

    N = q.shape[0]
    T_output = q.shape[1]
    T_input = k.shape[1]

    # change the shape
    # (N, T, h, d_k) -> (N, h, T, d_k)
    # in order for matrix multiply to work properly
    q = q.view(N, T_output, self.n_heads, self.d_k).transpose(1, 2)
    k = k.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)
    v = v.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)

    # Compute attention weights
    # (N, h, T, d_k) x (N, h, d_k, T) -> (N, h, T, T)
    attn_scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
    if pad_mask is not None:
      attn_scores = attn_scores.masked_fill(pad_mask[:, None, None, :] == 0, float('-inf'))

    if self.causal:
      attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T_output, :T_input] == 0, float('-inf'))

    attn_weights = F.softmax(attn_scores, dim=-1)

    # Compute attention weights values
    # (N, h, T, T) x (N, h, T, d_k) -> (N, h, T, d_k)
    A = attn_weights @ v

    # Reshape it back before final linear layer
    A = A.transpose(1, 2) # (N, T, h, d_k)
    A = A.contiguous().view(N, T_output, self.d_k * self.n_heads) # (N, T, h*d_k)

    # projection
    return self.fc(A)

In [8]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=256, dropout_prob=0.1) -> None:
    super().__init__()

    self.dropout = nn.Dropout(p=dropout_prob)

    position = torch.arange(max_len).unsqueeze(1)
    exp_term = torch.arange(0, d_model, 2)
    div_term = torch.exp(exp_term * (-math.log(10000.0)/d_model))
    pe = torch.zeros(1, max_len, d_model)
    pe[0, :, 0::2] = torch.sin(position * div_term)
    pe[0, :, 1::2] = torch.cos(position * div_term)
    self.register_buffer('pe', pe)

  def forward(self, x):
    # x.shape (N, T, D)
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

In [9]:
class EncoderBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.mha = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, x, mask=None):
    x = self.ln1(x + self.mha(x, x, x, mask))
    x = self.ln2(x + self.ann(x))
    x = self.dropout(x)
    return x

In [10]:
class Encoder(nn.Module):
  def __init__(self, mz_bins, max_len, d_k, d_model, n_heads, n_layers,
               #n_classes,
               dropout_prob) -> None:
    super().__init__()

    # self.embedding = nn.Embedding(vocab_size, d_model)
    self.embedding = PeakEmbedding(d_model, mz_bins)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    transformer_blocks = [
        EncoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.transformer_blocks = nn.Sequential(*transformer_blocks)
    self.ln = nn.LayerNorm(d_model)
    #self.fc = nn.Linear(d_model, n_classes)

  def forward(self, mz_idx, intensity, mask=None):
    x = self.embedding(mz_idx, intensity)
    x = self.pos_encoding(x)

    for block in self.transformer_blocks:
      x = block(x, mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    #x = x[:, 0, :]

    # Masked mean pooling — averages the encoder output over real peaks only, ignoring padding
    mask_expanded = mask.unsqueeze(-1).float()  # [B, L, 1]
    x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1)
      
    x = self.ln(x)
    #x = self.fc(x)
    return x

In [11]:
from rdkit.Chem import rdFingerprintGenerator
import rdkit.Chem as Chem

class MorganEncoder:
    def __init__(self, radius=2, n_bits=2048):
        self.gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    def __call__(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        fp = self.gen.GetFingerprintAsNumPy(mol)   # returns np.uint8 array directly
        return fp.astype(np.float32)

In [12]:
class ProjectionHead(nn.Module):
    def __init__(self, d_model=256, d_fp=2048):
        super().__init__()
        self.proj = nn.Linear(d_model, d_fp)

    def forward(self, x):
        return self.proj(x)

In [13]:
from torch.utils.data import Dataset

class SpectrumDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        sample = self.tokenizer(
            row['ms2_mzs'],
            row['ms2_normalized_intensities'],
        )
        if self.is_test:
            sample['molecule_id'] = row['molecule_id']
        else:
            sample['smiles'] = row['normalized_smiles']
            sample['inchikey14'] = row['inchikey14']
        return sample



In [14]:
def info_nce_loss(spec_proj, mol_emb, molecule_ids, temperature=0.07):
    # normalize
    spec_proj = F.normalize(spec_proj, dim=-1)   # [B, d_fp]
    mol_emb = F.normalize(mol_emb, dim=-1)       # [B, d_fp]

    # similarity matrix
    logits = spec_proj @ mol_emb.T / temperature  # [B, B]

    # mask same-molecule pairs (off-diagonal entries where molecule_ids match)
    B = len(molecule_ids)
    same = molecule_ids.unsqueeze(0) == molecule_ids.unsqueeze(1)   # [B, B], True where same molecule
    eye = torch.eye(B, dtype=torch.bool, device=same.device)
    mask_positives = same & ~eye   # True for "different spectrum, same molecule"
    logits = logits.masked_fill(mask_positives, float('-inf'))  # exclude from negatives

    labels = torch.arange(B, device=logits.device)
    # each row in logits is treated as a separate training example
    return F.cross_entropy(logits, labels)

In [15]:
# load train, split, load test

train = pd.read_parquet(
    '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet',
    columns=['inchikey14', 'ms2_mzs', 'ms2_normalized_intensities', 'normalized_smiles']
)

test = pd.read_parquet(
    '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet',
    columns=['molecule_id', 'ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz', 'adduct']
)

In [16]:
# import pandas as pd
# coconut = pd.read_parquet('/kaggle/input/datasets/aidensong123/casmi26-coconut-202609/coconut_structures.parquet')
# print(coconut.shape)
# print(coconut.columns.tolist())
# print(coconut.head())
# print(coconut['canonical_smiles'].isna().sum())  # count of missing SMILES

In [17]:
# instantiate and load checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = SpectrumTokenizer()
collate = SpectrumCollator(max_peaks=256)
me = MorganEncoder()

enc = Encoder(
    mz_bins=tok.mz_bins,
    max_len=256,
    d_k=64,
    d_model=256,
    n_heads=4,
    n_layers=4,
    dropout_prob=0.1,
).to(device)
proj = ProjectionHead().to(device)

matches = list(Path("/kaggle/input").rglob("checkpoint2m4.pt"))
print("checkpoints found:", matches)
state = torch.load(matches[0], map_location=device)
enc.load_state_dict(state['enc'])
proj.load_state_dict(state['proj'])
enc.eval()
proj.eval()

checkpoints found: [PosixPath('/kaggle/input/datasets/salutsirius/enveda-2m4-dataset-from-notebook02a72b3003/checkpoint2m4.pt')]


ProjectionHead(
  (proj): Linear(in_features=256, out_features=2048, bias=True)
)

In [18]:
# pool

import gc
from rdkit.Chem.Descriptors import ExactMolWt

# --- training molecules ---
train_mols = train.drop_duplicates('inchikey14')[['inchikey14', 'normalized_smiles']].reset_index(drop=True)
print(f'train molecules: {len(train_mols)}')
train_ik_list = train_mols['inchikey14'].tolist()
train_smiles_list = train_mols['normalized_smiles'].tolist()

# --- COCONUT ---
coconut = pd.read_parquet(
    '/kaggle/input/datasets/aidensong123/casmi26-coconut-202609/coconut_structures.parquet',
    columns=['canonical_smiles', 'inchikey']
).dropna(subset=['canonical_smiles', 'inchikey']).reset_index(drop=True)
print(f'coconut rows: {len(coconut)}')
coconut_ik_list = coconut['inchikey'].str[:14].tolist()
coconut_smiles_list = coconut['canonical_smiles'].tolist()

# --- combined pool (lists only) ---
all_ik_list = train_ik_list + coconut_ik_list
all_smiles_list = train_smiles_list + coconut_smiles_list
print(f'combined pool: {len(all_smiles_list)} molecules')

del coconut, train_mols
gc.collect()

# --- exact neutral mass per pool molecule (8 MB, not 8 GB) ---
pool_mass = np.array([
    ExactMolWt(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else np.nan
    for s in tqdm(all_smiles_list, desc='pool masses')
])
print(f'mass finite: {np.isfinite(pool_mass).mean():.3f}')

# --- sort by mass ---
order = np.argsort(np.where(np.isfinite(pool_mass), pool_mass, 1e18), kind='mergesort')
pool_mass_sorted = pool_mass[order]
pool_smiles_sorted = [all_smiles_list[i] for i in order]
pool_ik_sorted = [all_ik_list[i] for i in order]
ik_to_smiles = dict(zip(all_ik_list, all_smiles_list))

# free the unsorted copies
del pool_mass, all_ik_list
gc.collect()

def mass_window(target, ppm=10.0):
    lo = np.searchsorted(pool_mass_sorted, target * (1 - ppm / 1e6), 'left')
    hi = np.searchsorted(pool_mass_sorted, target * (1 + ppm / 1e6), 'right')
    return lo, hi

print('done')

train molecules: 275810
coconut rows: 738827
combined pool: 1014637 molecules


pool masses:   0%|          | 0/1014637 [00:00<?, ?it/s]

mass finite: 1.000
done


In [19]:
# ============================================================
# Channel 1 setup: spectral library + Numba entropy-similarity kernels
# Built ONCE from full train.parquet. Nothing is withheld.
# ============================================================
import pyarrow as pa
import pyarrow.parquet as pq
from numba import njit, prange

MZ_TOL       = 0.01
C1_INT_FLOOR = 0.002
C1_MAX_PEAKS = 256
C1_INT_POWER = 1.0
C1_ENT_W     = True
TAU          = 0.9999   # Channel 1 override threshold

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx: mx = it[i]
    if mx <= 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr: c += 1
    idx = np.empty(c, np.int64); j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i; j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c): v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk): k2[i] = idx[o[i]]
        k2.sort(); idx = k2; c = topk
    om = np.empty(c, np.float32); oi = np.empty(c, np.float32); s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v; s += v
    if s > 0:
        for i in range(c): oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0: S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w; s2 += oi[i]
            if s2 > 0:
                for i in range(c): oi[i] /= s2
    return om, oi

@njit(cache=True, fastmath=True)
def _ent(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        dd = qmz[i] - cmz[j]
        if dd < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif dd > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n: buf[b] = qp[i]; i += 1; b += 1
    while j < m: buf[b] = cp[j]; j += 1; b += 1
    for x in range(b): tot += buf[x]
    if tot <= 0: return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0: SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)

# ---- build library from FULL train.parquet (no withholding) ----
TRAIN_PATH = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet'
_tab = pq.read_table(
    TRAIN_PATH,
    columns=['inchikey14', 'adduct', 'precursor_mz',
             'ms2_mzs', 'ms2_normalized_intensities'],
)

_lib_prec = _tab.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
_lib_add  = np.asarray(_tab.column('adduct').cast(pa.string()).to_pylist(), dtype=object)
lib_ik    = np.asarray(_tab.column('inchikey14').cast(pa.string()).to_pylist(), dtype=str)

_lib_nm = neutral_mass(_lib_prec, _lib_add)

_ok = np.isfinite(_lib_nm)
lib_order     = np.argsort(np.where(_ok, _lib_nm, 1e18), kind='mergesort')
lib_nm_sorted = _lib_nm[lib_order]
lib_n_ok      = int(_ok.sum())

_mzc = _tab.column('ms2_mzs').combine_chunks()
_itc = _tab.column('ms2_normalized_intensities').combine_chunks()
lib_off = _mzc.offsets.to_numpy().astype(np.int64)
lib_mz  = _mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
lib_in  = _itc.values.to_numpy(zero_copy_only=False).astype(np.float32)

del _tab, _mzc, _itc
import gc; gc.collect()

print(f'[Channel 1] library spectra: {len(lib_off)-1:,}')
print(f'[Channel 1] library structures: {len(np.unique(lib_ik)):,}')
print(f'[Channel 1] finite-mass spectra: {lib_n_ok:,}')
print(f'[Channel 1] TAU = {TAU}')

def channel1_best_ik(target, spec_iter, ppm_win=10.0, ppm_fb=35.0):
    """Return (best_ik, best_sim) for this molecule; (None, 0.0) if nothing found."""
    lo = np.searchsorted(lib_nm_sorted[:lib_n_ok], target * (1 - ppm_win/1e6), 'left')
    hi = np.searchsorted(lib_nm_sorted[:lib_n_ok], target * (1 + ppm_win/1e6), 'right')
    cand = lib_order[lo:hi]
    if len(cand) == 0:
        lo = np.searchsorted(lib_nm_sorted[:lib_n_ok], target * (1 - ppm_fb/1e6), 'left')
        hi = np.searchsorted(lib_nm_sorted[:lib_n_ok], target * (1 + ppm_fb/1e6), 'right')
        cand = lib_order[lo:hi]
    if len(cand) == 0:
        return None, 0.0

    cand_ik = lib_ik[cand]
    hits = {}
    for mz, it in spec_iter:
        qm, qp = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                        C1_INT_FLOOR, C1_MAX_PEAKS, C1_INT_POWER, C1_ENT_W)
        if len(qm) == 0:
            continue
        for k in range(len(cand)):
            c = cand[k]
            a = lib_off[c]; b = lib_off[c+1]
            if b <= a: continue
            cm, cp = _clean(lib_mz[a:b], lib_in[a:b],
                            C1_INT_FLOOR, C1_MAX_PEAKS, C1_INT_POWER, C1_ENT_W)
            if len(cm) == 0: continue
            s = _ent(qm, qp, cm, cp, MZ_TOL)
            key = str(cand_ik[k])
            if s > hits.get(key, -1.0):
                hits[key] = float(s)

    if not hits:
        return None, 0.0
    best_ik = max(hits.items(), key=lambda x: x[1])
    return best_ik[0], best_ik[1]

[Channel 1] library spectra: 2,539,608
[Channel 1] library structures: 275,810
[Channel 1] finite-mass spectra: 2,537,394
[Channel 1] TAU = 0.9999


In [20]:
from collections import defaultdict

test['neutral_mass'] = neutral_mass(
    test['precursor_mz'].values.astype(np.float64),
    test['adduct'].values
)

mols = list(test.groupby('molecule_id', sort=False))

submission_rows = []
n_fallback = 0
n_c1_promoted = 0
n_c1_none = 0
all_c1_sims = []   # diagnostic: track every non-zero c1_sim

for gi, (mid, sub) in enumerate(tqdm(mols, desc='molecules')):
    nms = sub['neutral_mass'].values[np.isfinite(sub['neutral_mass'].values)]
    if len(nms) == 0:
        submission_rows.append({'molecule_id': mid, 'smiles': 'CCO'})
        continue
    target = float(np.median(nms))

    # ---------------- Channel 1 ----------------
    spec_iter = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
    c1_ik, c1_sim = channel1_best_ik(target, spec_iter)
    if c1_ik is None:
        n_c1_none += 1
    else:
        all_c1_sims.append(float(c1_sim))

    if gi < 5 or gi % 100 == 0:
        print(f'  diag {gi:>3} {mid}: c1_ik={c1_ik} c1_sim={c1_sim:.4f}')

    # ---------------- 0.189 path (unchanged) ----------------
    lo, hi = mass_window(target, ppm=10.0)
    if hi <= lo:
        lo, hi = mass_window(target, ppm=35.0)
        n_fallback += 1
    if hi <= lo:
        if c1_ik is not None and c1_sim > TAU:
            smi0 = ik_to_smiles.get(c1_ik, 'CCO')
            smiles_25 = [smi0] * 25
            n_c1_promoted += 1
        else:
            smiles_25 = ['CCO'] * 25
        submission_rows.append({'molecule_id': mid, 'smiles': ';'.join(smiles_25)})
        continue

    cand_iks = pool_ik_sorted[lo:hi]
    cand_fps = np.stack([
        me(s) if me(s) is not None else np.zeros(2048, np.float32)
        for s in pool_smiles_sorted[lo:hi]
    ]).astype('float32')
    cand_fps /= np.linalg.norm(cand_fps, axis=1, keepdims=True) + 1e-9

    spectrum_scores = defaultdict(float)
    for r in sub.itertuples():
        s = tok(r.ms2_mzs, r.ms2_normalized_intensities)
        mz_t = s['mz_idx'].unsqueeze(0).to(device)
        int_t = s['intensity'].unsqueeze(0).to(device)
        mask_t = torch.ones_like(mz_t, dtype=torch.bool, device=device)

        with torch.no_grad():
            q = proj(enc(mz_t, int_t, mask_t))
            q = F.normalize(q, dim=-1).cpu().numpy().astype('float32')

        sims = (cand_fps @ q.T).squeeze(-1)
        order_sims = np.argsort(-sims)
        for rank, idx in enumerate(order_sims, start=1):
            spectrum_scores[cand_iks[idx]] += 1.0 / rank

    top25 = sorted(spectrum_scores.items(), key=lambda x: -x[1])[:25]
    smiles_25 = [ik_to_smiles[ik] for ik, _ in top25]

    # ---------------- Channel 1 override ----------------
    if c1_ik is not None and c1_sim > TAU:
        c1_smiles = ik_to_smiles.get(c1_ik)
        if c1_smiles is not None:
            smiles_25 = [c1_smiles] + [s for s in smiles_25 if s != c1_smiles]
            smiles_25 = smiles_25[:25]
            n_c1_promoted += 1

    while len(smiles_25) < 25:
        smiles_25.append(smiles_25[-1] if smiles_25 else 'CCO')

    submission_rows.append({'molecule_id': mid, 'smiles': ';'.join(smiles_25)})

sub = pd.DataFrame(submission_rows)
print(f'fallback used: {n_fallback}/{len(mols)}')
print(f'Channel 1 promoted to rank 1: {n_c1_promoted}/{len(mols)}')
print(f'Channel 1 found no candidates: {n_c1_none}/{len(mols)}')

if len(all_c1_sims) > 0:
    arr = np.array(all_c1_sims)
    print(f'\nc1_sim distribution over {len(arr)} molecules with a hit:')
    for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
        print(f'  p{p:>2}: {np.percentile(arr, p):.4f}')
    for th in [0.85, 0.90, 0.93, 0.95, 0.97, 0.99]:
        print(f'  > {th:.2f}: {(arr > th).mean():.1%}  ({(arr > th).sum()} molecules)')
else:
    print('No c1_sim values captured — channel1_best_ik returned None everywhere.')

print(sub.shape)
print(sub.head())

molecules:   0%|          | 0/400 [00:00<?, ?it/s]

  diag   0 m_005e53: c1_ik=QCTQNXJUUXYSRK c1_sim=1.0000
  diag   1 m_006153: c1_ik=BKIHGKNEJKYYAC c1_sim=1.0000
  diag   2 m_00b5aa: c1_ik=RWCPIENWOXGKAV c1_sim=1.0000
  diag   3 m_01e922: c1_ik=KYMDZNBJUGJEMP c1_sim=1.0000
  diag   4 m_0259d4: c1_ik=MQCUYCZPGUNNLD c1_sim=1.0000
  diag 100 m_402004: c1_ik=TZDLDJWMNRFRHY c1_sim=1.0000
  diag 200 m_8b6b8e: c1_ik=PUXPJVBJQJCLRF c1_sim=1.0000
  diag 300 m_c9c302: c1_ik=PIVPHBVKQGQQFB c1_sim=1.0000
fallback used: 0/400
Channel 1 promoted to rank 1: 400/400
Channel 1 found no candidates: 0/400

c1_sim distribution over 400 molecules with a hit:
  p 1: 1.0000
  p 5: 1.0000
  p10: 1.0000
  p25: 1.0000
  p50: 1.0000
  p75: 1.0000
  p90: 1.0000
  p95: 1.0000
  p99: 1.0000
  > 0.85: 100.0%  (400 molecules)
  > 0.90: 100.0%  (400 molecules)
  > 0.93: 100.0%  (400 molecules)
  > 0.95: 100.0%  (400 molecules)
  > 0.97: 100.0%  (400 molecules)
  > 0.99: 100.0%  (400 molecules)
(400, 2)
  molecule_id                                             smiles


In [21]:
print(test['adduct'].value_counts())
print([a for a in test['adduct'].unique() if a not in ADDUCTS])

adduct
[M+H]+          959
[M-H]-          193
[M+CH2O2-H]-     31
[M+Na]+          22
[M+NH4]+          4
[M+K]+            2
[M+Cl]-           2
Name: count, dtype: int64
[]


In [22]:
# validate and write

# every test molecule must appear exactly once
assert set(sub['molecule_id']) == set(test['molecule_id']), "missing or extra molecule_id"

# no more than 25 semicolon-separated guesses
max_guesses = sub['smiles'].apply(lambda s: len(s.split(';'))).max()
assert max_guesses <= 25, f"too many guesses: {max_guesses}"

sub.to_csv('submission.csv', index=False)
print('written submission.csv')

written submission.csv


In [23]:
print(ik_to_smiles.get('QCTQNXJUUXYSRK'))
print(ik_to_smiles.get('BKIHGKNEJKYYAC'))
print(ik_to_smiles.get('RWCPIENWOXGKAV'))

CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1
COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1
COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC
